In [1]:
import pandas as pd

xls = pd.ExcelFile("index.xlsx")

print(xls.sheet_names)

['INDEX', 'Sheet2', 'Sheet3', 'Sheet1', 'Duplicates', 'final', 'Sheet4', 'Dupplicates']


In [2]:
final_df=pd.read_excel("index.xlsx",sheet_name='final')
final_df

,DATE,ORDER NO,book,PARTY NAME,CONTACT NO
0,04-07-2007,101,NaN,Dr.Rakesbhai Chokshi,9879599890
1,04-07-2007,102,NaN,kamleshbhai Pandiya,9825709204
2,04-07-2007,103,NaN,Samirbhai Tank,9879568265
3,09-07-2010,104,NaN,Chandrakantbhai Dhodakiya,NaN
4,04-07-2007,106,NaN,AbhaiBahi N Parmar,9426784608
...,...,...,...,...,...
12356,09-12-2025,21091,NaN,R.B. Parmar,9879579547
12357,09-12-2025,21092,NaN,Dhirubhai Dadhaniya,9824201022
12358,10-12-2025,21097,NaN,Umangkumar Meghani,9510049003
12359,11-12-2025,21098,NaN,Bipinbhai mehta,9824415507


In [3]:
duplicate_df=pd.read_excel('index.xlsx',sheet_name='Duplicates')
duplicate_df

,DATE,ORDER NO,PARTY NAME,CONTACT NO,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,9.7.07,110.0,Himanshubhai Khatri,9427728736,NaN,NaN,NaN,NaN
1,9.7.10,402.0,Surendrabhai Khatri,9427728736,NaN,NaN,NaN,NaN
2,17.7.07,136.0,Harendrabhai Relvani,9825604709,NaN,NaN,NaN,NaN
3,17.7.07,137.0,Dineshbhai Patoliya,9925012323,NaN,NaN,NaN,NaN
4,17.7.07,403.0,Jigneshbhai Patoliya,9925012323,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
887,22.3.23,19556.0,Payalben gondaliya,7046626798,NaN,NaN,NaN,NaN
888,5.12.2007,592.0,Maheshbhai Mehta Bombay,GHATKOPAR,NaN,NaN,NaN,NaN
889,5.12.2007,593.0,Rajniubhai Mehta(Rajkot),9427221135,NaN,NaN,NaN,NaN
890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(final_df.columns.tolist())
print()
print(duplicate_df.columns.tolist())

print(final_df.dtypes)
print()
print(duplicate_df.dtypes)

['DATE', 'ORDER NO', 'book', 'PARTY NAME', 'CONTACT NO']

['DATE', 'ORDER NO', 'PARTY NAME', 'CONTACT NO', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7']
DATE           object
ORDER NO        int64
book          float64
PARTY NAME        str
CONTACT NO     object
dtype: object

DATE           object
ORDER NO      float64
PARTY NAME        str
CONTACT NO     object
Unnamed: 4     object
Unnamed: 5     object
Unnamed: 6    float64
Unnamed: 7    float64
dtype: object


In [16]:
lookUp={}

for _,row in duplicate_df.iterrows():

    if pd.isna(row["ORDER NO"]):
        continue

    key=(
        str(row["PARTY NAME"]).strip().lower(),
        str(row["CONTACT NO"]).strip()
    )
    lookUp[key]=int(row["ORDER NO"])

len(lookUp)

890

In [13]:
row = final_df.iloc[0]

key = (
    str(row["PARTY NAME"]).strip().lower(),
    str(row["CONTACT NO"]).strip()
)

print(key)

print(
    lookUp.get(key)
)

('dr.rakesbhai chokshi', '9879599890')
None


In [15]:
count = 0

for _, row in final_df.iterrows():

    key = (
        str(row["PARTY NAME"]).strip().lower(),
        str(row["CONTACT NO"]).strip()
    )

    if key in lookUp:
        count += 1

print(
    f"Matches Found: {count}"
)

Matches Found: 946


In [17]:
updated=0

for index,row in final_df.iterrows():

    key=(
        str(row["PARTY NAME"]).strip().lower(),
        str(row["CONTACT NO"]).strip()
    )

    if key in lookUp:

        final_df.loc[
            index,
            "ORDER NO"
        ]=lookUp[key]

        updated+=1

print(
    f"updated {updated} rows"
)

updated 946 rows


In [19]:
final_df[
    final_df["PARTY NAME"]
    .str.contains(
        "surendrabhai",
        case=False,
        na=False
    )
]

,DATE,ORDER NO,book,PARTY NAME,CONTACT NO
9,2007-07-09 00:00:00,402,NaN,Surendrabhai Khatri,9427728736
936,11-06-2008,1305,NaN,Surendrabhai Pujara,9727568887
8308,23-12-2015,12230,NaN,Surendrabhai Rudhani,7201963323


In [20]:
before = len(final_df)

final_df = final_df.drop_duplicates(
    subset=[
        "PARTY NAME",
        "CONTACT NO"
    ],
    keep="first"
)

after = len(final_df)

print(
    f"Removed {before-after} rows"
)

Removed 97 rows


In [21]:
print(len(final_df))

12264


In [22]:
with pd.ExcelWriter(
    "index.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    final_df.to_excel(
        writer,
        sheet_name="final",
        index=False
    )

    duplicate_df.to_excel(
        writer,
        sheet_name="Duplicates",
        index=False
    )